# 03 ? RDKit Descriptor Modeling

## Legacy workflow reproduction

This notebook reproduces the audited Notebook 5 descriptor-based Random Forest and MLP analysis on the accepted current DILIst cohort. It reports 5-fold cross-validation and pooled out-of-fold results. Neither is independent test performance.

## 1. Research question and scope

How do the two fixed, audited Notebook 5 models perform on the current accepted cohort when represented by RDKit molecular descriptors?

This is a **legacy workflow reproduction**, not final model selection or final evaluation. It adds no models, feature selection, or hyperparameter search. Historical values and current-cohort values are reported separately. The historical source cohort is unavailable for row-level verification.

## 2. Cohort and provenance

The input is the frozen `DILIst_Final.csv`. Expected SHA-256: `4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f68790a07a231b03dc32c7`. The expected current cohort has 1,198 rows (class 0: 463; class 1: 735). RDKit parsing excludes malformed DILIST_ID 397 for descriptor modeling, leaving 1,197 valid structures (class 0: 462; class 1: 735).

The historical Notebook 5 results refer to 1,202 input rows and 1,201 valid structures. That historical source file is unavailable; the historical cohort is not treated as identical to this current cohort.

## 3. Data loading and frozen-file verification

This cell reads the dataset and audit reference files. It does not alter the dataset. Assertions stop the notebook if the accepted file's identity, schema, row count, or classes differ from the accepted record.

In [1]:
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Descriptors
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")  # matches the original Notebook 5 warning policy
RDLogger.DisableLog("rdApp.*")  # retain parser failures in the explicit table below

search_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (search_start, *search_start.parents)
                     if (p / "DILIst_Final.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate DILIst_Final.csv from the notebook working directory.")
DATA_PATH = PROJECT_ROOT / "DILIst_Final.csv"
AUDIT_CSV_PATH = PROJECT_ROOT / "results" / "legacy_notebook5_reproduction.csv"
AUDIT_JSON_PATH = PROJECT_ROOT / "results" / "legacy_notebook5_reproduction.json"
TARGET = "DILIst Classification "
SMILES_COL = "SMILES"
EXPECTED_SHA256 = "4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f68790a07a231b03dc32c7"
EXPECTED_COLUMNS = ["DILIST_ID", "CompoundName", TARGET, "Routs of Administration ", SMILES_COL]
EXPECTED_CLASS_COUNTS = {0: 463, 1: 735}


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

input_sha256 = sha256_file(DATA_PATH)
raw_df = pd.read_csv(DATA_PATH)
class_counts_input = {int(k): int(v) for k, v in raw_df[TARGET].value_counts().sort_index().items()}
input_schema = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": [str(raw_df[c].dtype) for c in raw_df.columns],
    "missing_count": [int(raw_df[c].isna().sum()) for c in raw_df.columns],
})

print("Input:", DATA_PATH.name)
print("SHA-256:", input_sha256)
print("Shape:", raw_df.shape)
print("Target:", repr(TARGET))
print("Class distribution:", class_counts_input)
print("RDKit:", rdBase.rdkitVersion, "| scikit-learn:", sklearn.__version__)
display(input_schema)

assert input_sha256 == EXPECTED_SHA256, "Accepted dataset hash mismatch; stop."
assert raw_df.shape == (1198, 5), f"Unexpected input shape: {raw_df.shape}"
assert list(raw_df.columns) == EXPECTED_COLUMNS
assert class_counts_input == EXPECTED_CLASS_COUNTS
assert int(raw_df[SMILES_COL].isna().sum()) == 0
assert AUDIT_CSV_PATH.is_file() and AUDIT_JSON_PATH.is_file(), "Expected accepted audit references are missing."


Input: DILIst_Final.csv
SHA-256: 4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f68790a07a231b03dc32c7
Shape: (1198, 5)
Target: 'DILIst Classification '
Class distribution: {0: 463, 1: 735}
RDKit: 2026.03.1 | scikit-learn: 1.8.0


,column,dtype,missing_count
0,DILIST_ID,int64,0
1,CompoundName,str,0
2,DILIst Classification,int64,0
3,Routs of Administration,str,257
4,SMILES,str,0


## 4. SMILES validity filtering

The original Notebook 5 calculates descriptors only for SMILES that RDKit parses. The accepted CSV remains unchanged; invalid rows are excluded from the in-memory model cohort only. The malformed record is reported by ID and name, without rewriting its source SMILES.

In [2]:
molecules = []
invalid_records = []
for row_position, row in raw_df.iterrows():
    try:
        mol = Chem.MolFromSmiles(str(row[SMILES_COL])) if pd.notna(row[SMILES_COL]) else None
    except Exception as exc:
        mol = None
        failure = f"{type(exc).__name__}: {exc}"
    else:
        failure = "Chem.MolFromSmiles returned None" if mol is None else None
    molecules.append(mol)
    if mol is None:
        invalid_records.append({
            "row_index_0based": int(row_position),
            "DILIST_ID": int(row["DILIST_ID"]),
            "CompoundName": row["CompoundName"],
            "failure": failure,
        })

valid_mask = np.asarray([mol is not None for mol in molecules], dtype=bool)
df_model = raw_df.loc[valid_mask].reset_index(drop=True)
mols_valid = [mol for mol in molecules if mol is not None]
y = df_model[TARGET].astype(int).to_numpy()
valid_class_counts = {int(k): int(v) for k, v in df_model[TARGET].value_counts().sort_index().items()}

print(f"Input rows: {len(raw_df)}")
print(f"Valid structures used for descriptor modeling: {len(df_model)}")
print(f"Invalid structures excluded: {len(invalid_records)}")
display(pd.DataFrame(invalid_records))
print("Valid-row class counts:", valid_class_counts)

assert len(df_model) == 1197
assert [x["DILIST_ID"] for x in invalid_records] == [397]
assert valid_class_counts == {0: 462, 1: 735}


Input rows: 1198
Valid structures used for descriptor modeling: 1197
Invalid structures excluded: 1


,row_index_0based,DILIST_ID,CompoundName,failure
0,386,397,nitroprusside,Chem.MolFromSmiles returned None


Valid-row class counts: {0: 462, 1: 735}


## 5. RDKit descriptor generation

The original Notebook 5 obtains descriptor names and functions from `rdkit.Chem.Descriptors.descList` and applies every function independently to each valid molecule. Each descriptor call has its own exception handler; an exception becomes NaN. No descriptor definition is substituted. RDKit version was not recorded in the original notebook; the version used in this execution is printed above.

In [3]:
desc_names = [name for name, _ in Descriptors.descList]
desc_funcs = [func for _, func in Descriptors.descList]
function_exceptions = {}

def calc_rdkit_descriptors(mol):
    values = []
    for name, func in zip(desc_names, desc_funcs):
        try:
            values.append(func(mol))
        except Exception as exc:
            function_exceptions[name] = function_exceptions.get(name, 0) + 1
            values.append(np.nan)
    return values

# Original workflow: calculate descriptors across the whole valid cohort before CV.
descriptor_rows = [calc_rdkit_descriptors(mol) for mol in mols_valid]
X_raw = pd.DataFrame(descriptor_rows, columns=desc_names)
print("Descriptor count before filtering:", X_raw.shape[1])
print("Descriptor function exceptions by name:", function_exceptions)
print("Descriptor names (original RDKit descList order):")
print(desc_names)

assert X_raw.shape == (1197, 217), f"Unexpected descriptor matrix shape: {X_raw.shape}"


Descriptor count before filtering: 217
Descriptor function exceptions by name: {}
Descriptor names (original RDKit descList order):
['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'AvgIpc', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'HallKierAlpha', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA

## 6. Descriptor quality and audited missing-value handling

The original order is preserved: numeric coercion (`errors='coerce'`), replace positive/negative infinity with NaN, replace finite absolute values greater than `1e12` with NaN, then drop all-NaN columns. These operations happen once on the full valid cohort before CV, as in the audited legacy workflow. The audit observed 260 initial NaN cells and 77 extreme-value cells converted to NaN; all 217 columns were retained. The checks below recompute those counts.

In [4]:
raw_nan_cells = int(X_raw.isna().sum().sum())
X_numeric = X_raw.apply(pd.to_numeric, errors="coerce")
coercion_new_nan_cells = int((X_numeric.isna() & X_raw.notna()).sum().sum())
infinite_cells = int(np.isinf(X_numeric.to_numpy(dtype=float)).sum())
X_finite = X_numeric.replace([np.inf, -np.inf], np.nan)
extreme_mask = X_finite.abs() > 1e12
extreme_cells = int(extreme_mask.sum().sum())
X_clean = X_finite.mask(extreme_mask, np.nan)
all_nan_columns = X_clean.columns[X_clean.isna().all()].tolist()
X_desc = X_clean.dropna(axis=1, how="all")
nan_cells_after_cleaning = int(X_desc.isna().sum().sum())

quality_summary = pd.DataFrame([
    {"check": "descriptors generated", "value": X_raw.shape[1]},
    {"check": "raw NaN cells before numeric coercion", "value": raw_nan_cells},
    {"check": "new NaN cells from nonnumeric coercion", "value": coercion_new_nan_cells},
    {"check": "infinite cells before replacement", "value": infinite_cells},
    {"check": "finite |value| > 1e12 cells converted to NaN", "value": extreme_cells},
    {"check": "all-NaN columns removed", "value": len(all_nan_columns)},
    {"check": "NaN cells before pipeline imputation", "value": nan_cells_after_cleaning},
    {"check": "descriptors retained", "value": X_desc.shape[1]},
])
display(quality_summary)
print("All-NaN columns:", all_nan_columns)

assert X_raw.shape[1] == 217
assert function_exceptions == {}
assert coercion_new_nan_cells == 0
assert infinite_cells == 0
assert extreme_cells == 77
assert len(all_nan_columns) == 0
assert X_desc.shape == (1197, 217)


,check,value
0,descriptors generated,217
1,raw NaN cells before numeric coercion,260
2,new NaN cells from nonnumeric coercion,0
3,infinite cells before replacement,0
4,finite |value| > 1e12 cells converted to NaN,77
5,all-NaN columns removed,0
6,NaN cells before pipeline imputation,337
7,descriptors retained,217


All-NaN columns: []


## 7. Legacy preprocessing and evaluation procedure

This reproduction deliberately keeps descriptor calculation, numeric conversion, infinity/extreme handling, and all-NaN-column filtering outside the CV folds because that is where the original Notebook 5 placed them. They are cohort-wide, unsupervised preprocessing steps and are not fold-local. Median imputation is inside each estimator pipeline; MLP scaling is also inside its pipeline, so these steps are fitted within each training fold.

The notebook uses `cross_validate` and, separately, `cross_val_predict(method='predict_proba')` with the same 5-fold shuffled `StratifiedKFold(random_state=42)`. Fold means/std and pooled OOF metrics are different summaries. OOF predictions are fold-held-out predictions, but CV/OOF remain internal resampling results, not independent test performance.

## 8. Fixed audited models

Only the original Notebook 5 configurations are used. There is no GridSearch, hyperparameter tuning, full-cohort refit, or model-file save.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "roc_auc", "f1"]

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

mlp_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=0.001,
        batch_size=32,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=42,
    )),
])

print("RF:", rf_model)
print("MLP:", mlp_model)


RF: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        min_samples_split=5, n_estimators=300,
                                        n_jobs=-1, random_state=42))])
MLP: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 MLPClassifier(alpha=0.001, batch_size=32, early_stopping=True,
                               hidden_layer_sizes=(256, 128), max_iter=500,
                               n_iter_no_change=30, random_state=42,
                               validation_fraction=0.15))])


## 9. Five-fold cross-validation

Fold-level scores are shown for each fixed estimator. Summary standard deviations use population standard deviation (`ddof=0`), matching the audit record. This table is **5-fold cross-validation performance**, not test performance.

In [6]:
def run_cv(name, estimator, X, y):
    scores = cross_validate(
        estimator, X, y, cv=cv, scoring=scoring, n_jobs=-1,
        return_train_score=False,
    )
    rows = []
    for fold_i in range(5):
        rows.append({
            "model": name,
            "fold": fold_i + 1,
            "accuracy": float(scores["test_accuracy"][fold_i]),
            "roc_auc": float(scores["test_roc_auc"][fold_i]),
            "f1": float(scores["test_f1"][fold_i]),
        })
    summary = {
        "model": name,
        "n_samples": len(y),
        "accuracy_mean": float(np.mean(scores["test_accuracy"])),
        "accuracy_std": float(np.std(scores["test_accuracy"], ddof=0)),
        "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
        "roc_auc_std": float(np.std(scores["test_roc_auc"], ddof=0)),
        "f1_mean": float(np.mean(scores["test_f1"])),
        "f1_std": float(np.std(scores["test_f1"], ddof=0)),
    }
    return rows, summary

rf_fold_rows, rf_cv_summary = run_cv("Random Forest", rf_model, X_desc, y)
mlp_fold_rows, mlp_cv_summary = run_cv("MLP", mlp_model, X_desc, y)
fold_results = pd.DataFrame(rf_fold_rows + mlp_fold_rows)
cv_summary = pd.DataFrame([rf_cv_summary, mlp_cv_summary])
display(fold_results)
display(cv_summary)


,model,fold,accuracy,roc_auc,f1
0,Random Forest,1,0.716667,0.711506,0.792683
1,Random Forest,2,0.679167,0.741935,0.752412
2,Random Forest,3,0.648536,0.675244,0.732484
3,Random Forest,4,0.656904,0.670105,0.743750
4,Random Forest,5,0.686192,0.724268,0.763407
5,MLP,1,0.641667,0.635579,0.729560
6,MLP,2,0.650000,0.694316,0.718121
7,MLP,3,0.640167,0.656389,0.732919
8,MLP,4,0.631799,0.663598,0.725000
9,MLP,5,0.648536,0.641452,0.760000


,model,n_samples,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std
0,Random Forest,1197,0.677493,0.023981,0.704612,0.027857,0.756947,0.020552
1,MLP,1197,0.642434,0.006531,0.658267,0.020635,0.733120,0.014326


## 10. Pooled out-of-fold metrics and error counts

The following predictions are generated separately with `cross_val_predict` and the same splitter configuration. Each row's probability is produced by a model that did not train on that row. Metrics are calculated once over pooled OOF probabilities/predictions. These remain OOF estimates, not independent test results.

In [7]:
rf_oof_probability = cross_val_predict(
    rf_model, X_desc, y, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]
mlp_oof_probability = cross_val_predict(
    mlp_model, X_desc, y, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]
rf_oof_prediction = (rf_oof_probability >= 0.5).astype(int)
mlp_oof_prediction = (mlp_oof_probability >= 0.5).astype(int)

def pooled_metrics(y_true, probability, prediction):
    return {
        "accuracy": float(accuracy_score(y_true, prediction)),
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "f1": float(f1_score(y_true, prediction)),
    }

oof_results = pd.DataFrame([
    {"model": "Random Forest", **pooled_metrics(y, rf_oof_probability, rf_oof_prediction)},
    {"model": "MLP", **pooled_metrics(y, mlp_oof_probability, mlp_oof_prediction)},
])
display(oof_results)

rf_error_mask = rf_oof_prediction != y
mlp_error_mask = mlp_oof_prediction != y
both_error_mask = rf_error_mask & mlp_error_mask
hard_union_mask = rf_error_mask | mlp_error_mask
hard_example_counts = pd.DataFrame([
    {"measure": "RF OOF errors at probability threshold 0.5", "count": int(rf_error_mask.sum())},
    {"measure": "MLP OOF errors at probability threshold 0.5", "count": int(mlp_error_mask.sum())},
    {"measure": "Both models incorrect", "count": int(both_error_mask.sum())},
    {"measure": "Row-wise union of either model incorrect", "count": int(hard_union_mask.sum())},
])
display(hard_example_counts)

# Keep the original Notebook 5 error table fields; include DILIST_ID for traceable current rows.
oof_comparison = df_model[["DILIST_ID", "CompoundName", SMILES_COL]].copy()
oof_comparison["y_true"] = y
oof_comparison["rf_pred"] = rf_oof_prediction
oof_comparison["rf_prob"] = rf_oof_probability
oof_comparison["nn_pred"] = mlp_oof_prediction
oof_comparison["nn_prob"] = mlp_oof_probability
oof_comparison["rf_correct"] = ~rf_error_mask
oof_comparison["nn_correct"] = ~mlp_error_mask
hard_examples = oof_comparison.loc[hard_union_mask].copy()
print("Hard-example definition: row-wise union of RF OOF incorrect OR MLP OOF incorrect at threshold 0.5.")
print("Each observation appears once; no independent test errors are represented.")
display(hard_examples.head(10))


,model,accuracy,roc_auc,f1
0,Random Forest,0.677527,0.704557,0.757233
1,MLP,0.642439,0.657736,0.733831


,measure,count
0,RF OOF errors at probability threshold 0.5,386
1,MLP OOF errors at probability threshold 0.5,428
2,Both models incorrect,288
3,Row-wise union of either model incorrect,526


Hard-example definition: row-wise union of RF OOF incorrect OR MLP OOF incorrect at threshold 0.5.
Each observation appears once; no independent test errors are represented.


,DILIST_ID,CompoundName,SMILES,y_true,rf_pred,rf_prob,nn_pred,nn_prob,rf_correct,nn_correct
3,4,chlorpheniramine,CN(C)CCC(C1=CC=C(C=C1)Cl)C2=CC=CC=N2,0,0,0.287354,1,0.781404,True,False
6,7,dopamine,C1=CC(=C(C=C1CCN)O)O,0,1,0.734547,1,0.609205,False,False
14,15,phenobarbital,CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2,1,1,0.559113,0,0.495122,True,False
16,17,rifampin,C[C@H]1/C=C/C=C(\C(=O)NC2=C(C(=C3C(=C2O)C(=C(C...,1,0,0.493396,1,0.626165,False,True
19,20,aciclovir,C1=NC2=C(N1COCCO)N=C(NC2=O)N,1,1,0.642675,0,0.389118,True,False
21,22,amineptine,C1CC2=CC=CC=C2C(C3=CC=CC=C31)NCCCCCCC(=O)O,1,1,0.539258,0,0.499261,True,False
24,25,amphotericin b,C[C@H]1/C=C/C=C/C=C/C=C/C=C/C=C/C=C/[C@@H](C[C...,1,0,0.261117,0,0.261400,False,False
28,29,aminohippurate,C1=CC(=CC=C1C(=O)NCC(=O)O)N,0,1,0.717250,1,0.870293,False,False
33,34,carboplatin,C1CC(C1)(C(=O)O)C(=O)O.[NH2-].[NH2-].[Pt+2],1,0,0.459610,0,0.337899,False,False
35,36,chlorcyclizine,CN1CCN(CC1)C(C2=CC=CC=C2)C3=CC=C(C=C3)Cl,1,1,0.543131,0,0.442410,True,False


## 11. Current-cohort reference comparison

The audited reproduction CSV is loaded as a reference, then compared to the newly computed values above. The reference numbers are not substituted for the recomputed metrics. Differences are reported numerically; when they exceed the display tolerance, the cause is not assigned without evidence.

In [8]:
audit_csv = pd.read_csv(AUDIT_CSV_PATH)
audit_json = json.loads(AUDIT_JSON_PATH.read_text(encoding="utf-8"))

model_name_map = {
    "Random Forest": "RandomForest (RDKit descriptors)",
    "MLP": "Neural Network MLP (RDKit descriptors)",
}
actual_summaries = {"Random Forest": rf_cv_summary, "MLP": mlp_cv_summary}
actual_oof = {
    "Random Forest": pooled_metrics(y, rf_oof_probability, rf_oof_prediction),
    "MLP": pooled_metrics(y, mlp_oof_probability, mlp_oof_prediction),
}
comparison_rows = []
for short_name, audit_name in model_name_map.items():
    ref = audit_csv.loc[audit_csv["model"].eq(audit_name)]
    if len(ref) != 1:
        raise ValueError(f"Expected one audit reference row for {audit_name}.")
    ref = ref.iloc[0]
    cv_actual = actual_summaries[short_name]
    for metric in ["accuracy", "roc_auc", "f1"]:
        actual_value = cv_actual[f"{metric}_mean"]
        reference_value = float(ref[f"cv_{metric}_mean"])
        comparison_rows.append({
            "model": short_name, "summary": "CV mean", "metric": metric,
            "recomputed": actual_value, "audited_current_reference": reference_value,
            "difference": actual_value - reference_value,
        })
        actual_value = actual_oof[short_name][metric]
        reference_value = float(ref[f"oof_{metric}"])
        comparison_rows.append({
            "model": short_name, "summary": "pooled OOF", "metric": metric,
            "recomputed": actual_value, "audited_current_reference": reference_value,
            "difference": actual_value - reference_value,
        })

current_reference_comparison = pd.DataFrame(comparison_rows)
display(current_reference_comparison)

tolerance = 1e-6
large_differences = current_reference_comparison.loc[
    current_reference_comparison["difference"].abs() > tolerance
]
if large_differences.empty:
    print(f"All recomputed current metrics agree with the audited reference within {tolerance:g}.")
else:
    print("One or more metrics differ from the audited reference by more than the display tolerance.")
    print("Cause not established unless supported by the recorded environment/evidence.")
    display(large_differences)


,model,summary,metric,recomputed,audited_current_reference,difference
0,Random Forest,CV mean,accuracy,0.677493,0.677493,0.0
1,Random Forest,pooled OOF,accuracy,0.677527,0.677527,0.0
2,Random Forest,CV mean,roc_auc,0.704612,0.704612,0.0
3,Random Forest,pooled OOF,roc_auc,0.704557,0.704557,0.0
4,Random Forest,CV mean,f1,0.756947,0.756947,0.0
5,Random Forest,pooled OOF,f1,0.757233,0.757233,0.0
6,MLP,CV mean,accuracy,0.642434,0.642434,0.0
7,MLP,pooled OOF,accuracy,0.642439,0.642439,0.0
8,MLP,CV mean,roc_auc,0.658267,0.658267,0.0
9,MLP,pooled OOF,roc_auc,0.657736,0.657736,0.0


All recomputed current metrics agree with the audited reference within 1e-06.


## 12. Historical Notebook 5 results ? reference only

These values come from the original Notebook 5's historical 1,202-row input / 1,201 valid structures. The source file and row-level cohort are unavailable for direct verification. The original notebook did not report historical CV standard deviations; those remain unavailable. Historical values are not current-cohort results.

In [9]:
historical_rows = []
for short_name, audit_name in model_name_map.items():
    ref = audit_csv.loc[audit_csv["model"].eq(audit_name)].iloc[0]
    historical_rows.append({
        "model": short_name,
        "historical_n_samples": int(ref["historical_n_samples"]),
        "cv_accuracy": float(ref["historical_cv_accuracy"]),
        "cv_roc_auc": float(ref["historical_cv_roc_auc"]),
        "cv_f1": float(ref["historical_cv_f1"]),
        "cv_std": "not reported",
        "pooled_oof_accuracy": float(ref["historical_oof_accuracy"]),
        "pooled_oof_roc_auc": float(ref["historical_oof_roc_auc"]),
        "pooled_oof_f1": float(ref["historical_oof_f1"]),
    })
historical_results = pd.DataFrame(historical_rows)
display(historical_results)

historical_mlp = historical_results.loc[historical_results["model"].eq("MLP")].iloc[0]
historical_mlp_auc_gap = float(historical_mlp["cv_roc_auc"] - historical_mlp["pooled_oof_roc_auc"])
current_mlp_auc_gap = float(mlp_cv_summary["roc_auc_mean"] - actual_oof["MLP"]["roc_auc"])
print(f"Historical MLP CV mean ROC-AUC minus pooled OOF ROC-AUC: {historical_mlp_auc_gap:.6f}")
print(f"Current MLP CV mean ROC-AUC minus pooled OOF ROC-AUC: {current_mlp_auc_gap:.6f}")
print("The historical MLP difference is preserved; cause not established.")


,model,historical_n_samples,cv_accuracy,cv_roc_auc,cv_f1,cv_std,pooled_oof_accuracy,pooled_oof_roc_auc,pooled_oof_f1
0,Random Forest,1201,0.678607,0.714336,0.757203,not reported,0.678601,0.713839,0.757538
1,MLP,1201,0.661940,0.674838,0.739388,not reported,0.661948,0.661949,0.739744


Historical MLP CV mean ROC-AUC minus pooled OOF ROC-AUC: 0.012889
Current MLP CV mean ROC-AUC minus pooled OOF ROC-AUC: 0.000531
The historical MLP difference is preserved; cause not established.


## 13. Current RF-versus-MLP comparison

Within this specific current-cohort legacy reproduction, compare each model's fold CV mean ? standard deviation and pooled OOF metrics. This comparison does not select the project's final model. The historical reference table above is separate and is not pooled into this current comparison.

In [10]:
current_model_comparison = cv_summary[[
    "model", "n_samples", "accuracy_mean", "accuracy_std",
    "roc_auc_mean", "roc_auc_std", "f1_mean", "f1_std"
]].merge(oof_results, on="model", suffixes=("_cv_summary", "_oof"))
display(current_model_comparison)
print("These current-cohort CV/OOF metrics are internal resampling results, not independent test performance.")
print("No overall project model is selected in this notebook.")


,model,n_samples,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,accuracy,roc_auc,f1
0,Random Forest,1197,0.677493,0.023981,0.704612,0.027857,0.756947,0.020552,0.677527,0.704557,0.757233
1,MLP,1197,0.642434,0.006531,0.658267,0.020635,0.733120,0.014326,0.642439,0.657736,0.733831


These current-cohort CV/OOF metrics are internal resampling results, not independent test performance.
No overall project model is selected in this notebook.


## 14. Interpretation

The two fixed models can be compared within this cohort and protocol using the reported CV and pooled OOF metrics. The Random Forest has higher CV and pooled OOF ROC-AUC in the audited current reproduction; this is a statement about this analysis only. It is not a final project model decision and does not establish independent generalization.

## 15. Methodological limitations

- There is no independent test set. CV means and pooled OOF metrics are not independent test performance.
- Descriptor generation, numeric coercion, infinity/extreme handling, and all-NaN-column filtering occur before CV across the complete current cohort. These steps are unsupervised but not fold-local.
- Median imputation is fitted inside each CV training fold; MLP standardization is also fitted inside each training fold.
- `cross_val_predict` gives genuinely out-of-fold probabilities for each row. Pooled OOF metrics and fold-mean CV metrics are different aggregations and may differ.
- The RF parameter configuration has earlier legacy selection history. Notebook 5 has no nested CV accounting for that selection.
- The historical 1,202-row source cannot be verified row by row and is not treated as identical to the current cohort.
- The historical MLP CV ROC-AUC (0.674838) differs from pooled OOF ROC-AUC (0.661949); the discrepancy is retained and its cause is not established.
- The 0.5 threshold is used as in the original procedure; it is not tuned here.

## 16. Reproducibility summary and protected-file checks

The final cell rechecks the accepted dataset and confirms that this execution did not change Notebook 01, Notebook 02, or the archived original Notebook 5. It also records the software versions used for this reproduction. This notebook performs no model refit on the full cohort and saves no model artifact.

In [11]:
# Recheck all protected inputs and notebooks after modeling.
ORIGINAL_NB5_PATH = next(PROJECT_ROOT.glob("5.*.ipynb"), None)
if ORIGINAL_NB5_PATH is None:
    raise FileNotFoundError("Could not locate the original Notebook 5 for the final preservation check.")
NOTEBOOK01_PATH = PROJECT_ROOT / "notebooks" / "final" / "01_dataset_construction.ipynb"
NOTEBOOK02_PATH = PROJECT_ROOT / "notebooks" / "final" / "02_morgan_fingerprint_analysis.ipynb"
EXPECTED_PROTECTED_HASHES = {
    "DILIst_Final.csv": EXPECTED_SHA256,
    "Notebook 01": "28b6c13409d1fc639ae17c6c0bca742a883cff29cc0de5acd3838d158f8da865",
    "Notebook 02": "55e2ab69669f62f4bdab82e7d938c3b13e27e8948ed9a2028e06a5bb84971943",
    "original Notebook 5": "049e55549213b90d49d7b45af86e759890c0f00f6909fbb8039146b303b42925",
}
protected_paths = {
    "DILIst_Final.csv": DATA_PATH,
    "Notebook 01": NOTEBOOK01_PATH,
    "Notebook 02": NOTEBOOK02_PATH,
    "original Notebook 5": ORIGINAL_NB5_PATH,
}
protected_check = pd.DataFrame([
    {
        "file": name,
        "sha256": sha256_file(path),
        "expected_sha256": EXPECTED_PROTECTED_HASHES[name],
        "unchanged": sha256_file(path) == EXPECTED_PROTECTED_HASHES[name],
    }
    for name, path in protected_paths.items()
])
display(protected_check)
assert protected_check["unchanged"].all(), "A protected file hash changed; investigate before accepting this run."

print("Notebook execution status: completed without cell errors (when run top to bottom).")
print("No test set, model search, final model selection, full-data refit, or model artifact save was performed.")
print("Reproduction differences beyond tolerance:", len(large_differences))


,file,sha256,expected_sha256,unchanged
0,DILIst_Final.csv,4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f687...,4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f687...,True
1,Notebook 01,28b6c13409d1fc639ae17c6c0bca742a883cff29cc0de5...,28b6c13409d1fc639ae17c6c0bca742a883cff29cc0de5...,True
2,Notebook 02,55e2ab69669f62f4bdab82e7d938c3b13e27e8948ed9a2...,55e2ab69669f62f4bdab82e7d938c3b13e27e8948ed9a2...,True
3,original Notebook 5,049e55549213b90d49d7b45af86e759890c0f00f6909fb...,049e55549213b90d49d7b45af86e759890c0f00f6909fb...,True


Notebook execution status: completed without cell errors (when run top to bottom).
No test set, model search, final model selection, full-data refit, or model artifact save was performed.
Reproduction differences beyond tolerance: 0
